In [1]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

# clone + install: https://github.com/Beater-221E/llm4rec-bias-Integrated
REPO_OWNER = "Beater-221E"
REPO_NAME = "llm4rec-bias-Integrated"
BRANCH = "distill"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
FRESH_CLONE = False
INSTALL_DEPS = True
CONDA_ENV = "bias"
TORCH_INDEX = "https://download.pytorch.org/whl/cu121"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return "COLAB_RELEASE_TAG" in os.environ


def _looks_like_repo(p: Path) -> bool:
    return (p / "src" / "llm4rec").is_dir() and (p / "run.sh").is_file() and (p / "prepare.sh").is_file()


def _find_existing() -> Path | None:
    env = os.environ.get("LLM4REC_ROOT")
    cands = [Path(env)] if env else []
    here = Path.cwd().resolve()
    cands += [here, here.parent, Path("/content") / REPO_NAME]
    for p in [here, *here.parents]:
        cands.append(p)
        if len(cands) > 24:
            break
    seen: set[Path] = set()
    for c in cands:
        try:
            c = c.resolve()
        except OSError:
            continue
        if c in seen:
            continue
        seen.add(c)
        if _looks_like_repo(c):
            return c
    return None


IN_COLAB = _in_colab()
ROOT = _find_existing()
if FRESH_CLONE and ROOT is not None and IN_COLAB:
    shutil.rmtree(ROOT)
    ROOT = None

if ROOT is None:
    dest = Path("/content" if IN_COLAB else Path.cwd()) / REPO_NAME
    dest.parent.mkdir(parents=True, exist_ok=True)
    clone_url = REPO_URL
    if GITHUB_TOKEN:
        clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, str(dest)])
    ROOT = dest.resolve()

os.chdir(ROOT)
os.environ["LLM4REC_ROOT"] = str(ROOT)
src = str(ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)
os.environ["PYTHONPATH"] = src + (":" + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")

# Colab：data / SID / runs（SFT、eval、后训练）写到 Drive，断线可续
DRIVE_FOLDER = "llm4rec-bias/minionerec_sda_ml100k"
PERSIST_NAMES = ("data", "artifacts", "runs")


def _mount_drive() -> Path | None:
    if not IN_COLAB:
        return None
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    root = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    root.mkdir(parents=True, exist_ok=True)
    return root


def _bind_persist(repo: Path, drive_root: Path) -> None:
    for name in PERSIST_NAMES:
        local = repo / name
        remote = drive_root / name
        remote.mkdir(parents=True, exist_ok=True)
        if local.is_symlink():
            if local.resolve() == remote.resolve():
                continue
            local.unlink()
        elif local.exists():
            for child in local.iterdir():
                dest = remote / child.name
                if dest.exists():
                    continue
                shutil.move(str(child), str(dest))
            if local.exists():
                shutil.rmtree(local)
        local.symlink_to(remote, target_is_directory=True)
        print(f"[persist] {name} → {remote}")


DRIVE_ROOT = _mount_drive()
if DRIVE_ROOT is not None:
    _bind_persist(ROOT, DRIVE_ROOT)
    os.environ["LLM4REC_DRIVE"] = str(DRIVE_ROOT)
    print(f"[persist] Drive: {DRIVE_ROOT}")
    subprocess.call(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", BRANCH])

if INSTALL_DEPS:
    py = sys.executable
    subprocess.check_call([py, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
    try:
        import torch  # noqa: F401
    except ImportError:
        subprocess.check_call(
            [py, "-m", "pip", "install", "-q", "torch", "--index-url", TORCH_INDEX]
        )
    subprocess.check_call([py, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
    subprocess.check_call([py, "-m", "pip", "install", "-q", "-e", str(ROOT)])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[persist] Drive: /content/drive/MyDrive/llm4rec-bias/minionerec_sda_ml100k


In [2]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any

ROOT = Path(os.environ.get("LLM4REC_ROOT") or Path.cwd()).resolve()
os.chdir(ROOT)
SRC = str(ROOT / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# EXP 用 amazon 底座；必须整组改成 ml-100k，否则 SID 指纹对不上
DATA_OVERRIDES = [
    "data.name=movielens",
    "data.variant=ml-100k",
    "data.category=ml-100k",
    "data.raw_dir=data/raw/movielens",
    "data.processed_dir=data/processed/movielens",
    "data.category_prompt=movies",
    "data.item_text_fields=[title,categories]",
    "data.item_text_max_chars=256",
]
# Colab 上从 Drive 找已有 run；本机仍可用下面两个 hint
_ON_DRIVE = bool(os.environ.get("LLM4REC_DRIVE"))
CFG: dict[str, Any] = {
    "exp": "minionerec_qwen05b_amazon",
    "gpus": "0",
    "conda_env": "" if _ON_DRIVE else "bias",
    "wandb_mode": "offline",
    "execute": True,
    "skip_if_done": True,
    # 旧 074107 是 chat+titles / 2epoch / 1e-5 / K=512，必须重训。
    # 新 SFT 跑完后改回 False。
    "force_sft": True,
    # True = 忽略已有 eval_1.json，按新的 sft/best（没有则 final）重评
    "force_eval": True,
    "overrides": list(DATA_OVERRIDES),
    "sft_checkpoint": "" if _ON_DRIVE else "runs/movielens_ml-100k/minionerec/qwen2.5-0.5b-instruct/seed_42/20260822_033438/sft/final",
    "distill_run_hint": "" if _ON_DRIVE else "runs/movielens_ml-100k/minionerec/qwen2.5-0.5b-instruct/seed_42/20260822_052807",
}

RUNS_GLOB = "runs/movielens_ml-100k/minionerec/qwen2.5-0.5b-instruct/seed_42/*"


In [3]:
def _gpu_mem_gb() -> float:
    try:
        import torch

        if torch.cuda.is_available():
            return torch.cuda.get_device_properties(0).total_memory / (1024**3)
    except Exception:
        pass
    return 0.0


# yaml 默认按 V100-16G（SFT micro=2 → accum=32）。A100-80G：SFT 拉大微批。
# Distill 不能照搬：16 prompt × 8 SID sample = 128 条序列一次反传，
# SID 扩词表 logits 会把 80G 打满。微批改 2，全局 batch 仍是 128。
_VRAM_GB = _gpu_mem_gb()
_A100 = _VRAM_GB >= 70
print(f"[hw] gpu_mem={_VRAM_GB:.1f} GiB  profile={'A100-80G' if _A100 else 'default'}")

# 训练配置覆盖：None / 空 dict = 沿用 yaml。改成具体值后会传给 prepare.sh / run.sh。
# 也可在 EXTRA_OVERRIDES 里直接写 dotted key，例如 "train.distill.epochs=2"。
TRAIN_OVERRIDES: dict[str, Any] = {
    # "seed": 42,
    "train": {
        "sft": {
            "recipe": "minionerec_reference",
            "objectives": ["sid_sft", "sid_item_feat", "fusion_seqrec"],
            "prompt_style": "minionerec_alpaca",
            "epochs": 8,
            "learning_rate": 3.0e-4,
            "lr_scheduler_type": "linear",
            "warmup_ratio": 0.03,
            "global_batch_size": 256 if _A100 else 64,
            "per_device_batch_size": 32 if _A100 else 2,
            "preferred_per_device_batch_size": 32 if _A100 else 2,
            "eval_steps": 200,
            "load_best_model_at_end": True,
        },
        "transition": {
            # "epochs": 10,
            "batch_size": 4096 if _A100 else 256,
            # "learning_rate": 0.001,
            # "hidden_dim": 256,
            # "embedding_dim": 128,
        },
        "distill": {
            # "epochs": 2,
            # "learning_rate": 1.0e-5,
            # "hard_weight": 0.5,
            # "samples_per_prompt": 8,
            # "exposure_weight": 0.1,
            # "probability_support": "full_vocab",
            "global_batch_size": 128,
            "per_device_batch_size": 2 if _A100 else 4,
            "preferred_per_device_batch_size": 2 if _A100 else 4,
            "catalog_chunk_size": 1024 if _A100 else 256,
            # "logging_steps": 10,
            "eval_steps": 200,
        },
    },
    "hardware": {
        "precision": "bf16" if _A100 else "auto",
        "gradient_checkpointing": True,
        "activation_checkpointing": "auto",
        "memory": "auto",
    },
    "evaluation": {
        # "top_k": [1, 5, 10, 20],
        # "max_examples": None,
        "batch_size": 16 if _A100 else 8,
    },
    "decoder": {
        # "num_beams": 20,
    },
}

EXTRA_OVERRIDES: list[str] = [
    "checkpoint.save_steps=200",
    "checkpoint.save_total_limit=4",
    "checkpoint.save_best=true",
    # A100 上 compile=auto 会开 inductor；变长 padding + SID 扩词表会在 backward 崩
    "optimization.compile.enabled=false",
    # 官方 MiniOneRec 码本 256；旧 ce6aaef8492e 是 512，hash 会变，必须重建 SID
    "sid.codebook_size=256",
    # "train.distill.samples_per_prompt=4",
]


def _flatten_overrides(prefix: str, value: Any, out: list[str]) -> None:
    if isinstance(value, dict):
        for key, item in value.items():
            nxt = f"{prefix}.{key}" if prefix else str(key)
            _flatten_overrides(nxt, item, out)
        return
    if value is None:
        return
    if isinstance(value, bool):
        out.append(f"{prefix}={'true' if value else 'false'}")
    elif isinstance(value, (list, tuple)):
        out.append(f"{prefix}=[{','.join(str(x) for x in value)}]")
    else:
        out.append(f"{prefix}={value}")


train_override_args: list[str] = []
_flatten_overrides("", TRAIN_OVERRIDES, train_override_args)
train_override_args.extend(s.strip() for s in EXTRA_OVERRIDES if str(s).strip() and not str(s).strip().startswith("#"))

seen: set[str] = set()
merged: list[str] = []
for item in [*DATA_OVERRIDES, *train_override_args]:
    key = item.split("=", 1)[0]
    if key in seen:
        merged[:] = [x for x in merged if x.split("=", 1)[0] != key]
    seen.add(key)
    merged.append(item)

CFG["overrides"] = merged


[hw] gpu_mem=79.3 GiB  profile=A100-80G


In [4]:
import json

def _is_ckpt(path: Path) -> bool:
    if not path.is_dir():
        return False
    return any(
        (path / name).exists()
        for name in ("config.json", "model.safetensors", "pytorch_model.bin", "model.safetensors.index.json")
    )


def find_latest(pattern: str) -> Path | None:
    hits = sorted(ROOT.glob(pattern), key=lambda p: p.stat().st_mtime if p.exists() else 0)
    return hits[-1] if hits else None


def _latest_stage_ckpt(stage: str) -> Path | None:
    for pattern in (f"{RUNS_GLOB}/{stage}/final", f"{RUNS_GLOB}/{stage}/best"):
        hit = find_latest(pattern)
        if hit is not None and _is_ckpt(hit):
            return hit
    mids = sorted(
        ROOT.glob(f"{RUNS_GLOB}/{stage}/checkpoint-*"),
        key=lambda p: p.stat().st_mtime if p.exists() else 0,
    )
    for path in reversed(mids):
        if _is_ckpt(path):
            return path
    return None


def resolve_sft_final() -> Path | None:
    """只认完整 sft/final。中间 checkpoint-* / best 不算训完。"""
    hinted = CFG.get("sft_checkpoint")
    if hinted:
        p = Path(hinted)
        if not p.is_absolute():
            p = ROOT / p
        if _is_ckpt(p) and p.name == "final":
            return p
    hit = find_latest(f"{RUNS_GLOB}/sft/final")
    if hit is not None and _is_ckpt(hit):
        return hit
    return None


def resolve_sft_eval_ckpt() -> Path | None:
    """评测优先 sft/best（val loss 最低），否则 sft/final。"""
    final = resolve_sft_final()
    if final is None:
        return None
    best = final.parent / "best"
    if _is_ckpt(best):
        return best
    return final


def _norm_ckpt(path: str | Path) -> str:
    p = Path(path)
    try:
        p = p.resolve()
    except OSError:
        pass
    return str(p).rstrip("/")


def find_eval_json(ckpt: Path | None) -> Path | None:
    """eval 会新开 run 目录；按 checkpoint 字段找回 eval_1.json。"""
    if ckpt is None:
        return None
    host = ckpt.parent.parent / "eval" / "eval_1.json"
    if host.is_file():
        return host
    wanted = {_norm_ckpt(ckpt)}
    for sibling in (ckpt.parent / "final", ckpt.parent / "best"):
        if _is_ckpt(sibling):
            wanted.add(_norm_ckpt(sibling))
    hits: list[Path] = []
    for path in ROOT.glob(f"{RUNS_GLOB}/eval/eval_1.json"):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        listed = str(payload.get("checkpoint") or "")
        if listed and _norm_ckpt(listed) in wanted:
            hits.append(path)
    if hits:
        return max(hits, key=lambda p: p.stat().st_mtime)
    return None


def attach_eval_json(ckpt: Path, eval_json: Path) -> Path:
    dest = ckpt.parent.parent / "eval" / "eval_1.json"
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.resolve() != eval_json.resolve():
        dest.write_text(eval_json.read_text(encoding="utf-8"), encoding="utf-8")
    return dest


def _ensure_sid_eval_generate_patch() -> None:
    """Colab clone 若还没 pull 到本次修复，就地清掉 Qwen instruct 的 generate 默认值。"""
    path = ROOT / "src/llm4rec/sid/constraint.py"
    if not path.is_file():
        return
    text = path.read_text(encoding="utf-8")
    if "repetition_penalty" in text and "user history" in text:
        return
    hook = "    gc.eos_token_id = int(eos_id)\n    gc.pad_token_id = int(eos_id)\n"
    extra = (
        "    gc.eos_token_id = int(eos_id)\n"
        "    gc.pad_token_id = int(eos_id)\n"
        "    gc.do_sample = False\n"
        "    for key, value in (\n"
        '        ("repetition_penalty", 1.0),\n'
        '        ("no_repeat_ngram_size", 0),\n'
        '        ("temperature", 1.0),\n'
        '        ("top_p", 1.0),\n'
        '        ("top_k", 0),\n'
        "    ):\n"
        "        if hasattr(gc, key):\n"
        "            try:\n"
        "                setattr(gc, key, value)\n"
        "            except (TypeError, ValueError):\n"
        "                pass\n"
    )
    if hook not in text:
        print("[patch] constraint.py 没有预期锚点，跳过")
        return
    path.write_text(text.replace(hook, extra, 1), encoding="utf-8")
    print("[patch] 已清除 Qwen instruct generate 默认值（repetition_penalty/top_k）")


_ensure_sid_eval_generate_patch()


def _assert_sft_recipe_src() -> None:
    """Colab 若还在旧 clone，官方 mix / Alpaca eval 不会生效。"""
    import inspect

    try:
        from llm4rec.data.examples import sid_seqrec_example
        from llm4rec.data.minionerec_sft import uses_reference_sft
    except ImportError as exc:
        raise RuntimeError(
            "当前 src 缺少 SFT recipe 修复。请同步本仓库最新源码后再跑 "
            "prepare / SFT / eval，不要继续评 074107。"
        ) from exc
    if "prompt_style" not in inspect.signature(sid_seqrec_example).parameters:
        raise RuntimeError(
            "sid_seqrec_example 没有 prompt_style。请同步最新 src，不要用旧 clone。"
        )
    if not uses_reference_sft({"experiment": {"route": "minionerec"}}):
        raise RuntimeError("uses_reference_sft 默认应为 True")


_assert_sft_recipe_src()


def resolve_distill_final() -> Path | None:
    hint = CFG.get("distill_run_hint")
    if hint:
        p = Path(hint)
        if not p.is_absolute():
            p = ROOT / p
        for name in ("final", "best"):
            cand = p / "distill" / name
            if _is_ckpt(cand):
                return cand
    return _latest_stage_ckpt("distill")


def _wanted_sid_codebook() -> int:
    for item in CFG.get("overrides") or []:
        if str(item).startswith("sid.codebook_size="):
            try:
                return int(str(item).split("=", 1)[1])
            except ValueError:
                break
    return 256


def sid_ready() -> bool:
    """只认当前 codebook（默认 256）。旧 K=512 产物不能 skip。"""
    wanted = _wanted_sid_codebook()
    for path in ROOT.glob("artifacts/sid/movielens_ml-100k/*/manifest.json"):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if int(payload.get("codebook_size") or 0) == wanted:
            return True
    return False


def processed_ready() -> bool:
    p = ROOT / "data/processed/movielens/ml-100k/interactions.jsonl"
    return p.is_file()


def run_script(
    script: str,
    *extra: str,
    env_extra: dict[str, str] | None = None,
    execute: bool | None = None,
) -> int:
    execute = CFG["execute"] if execute is None else execute
    env = os.environ.copy()
    env["EXP"] = str(CFG["exp"])
    env["GPUS"] = str(CFG["gpus"])
    env["CONDA_ENV"] = str(CFG["conda_env"])
    env["WANDB_MODE"] = str(CFG["wandb_mode"])
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    env.pop("STAGES", None)
    env.pop("RESUME_FROM", None)
    if env_extra:
        env.update({k: v for k, v in env_extra.items() if v})

    cmd = ["bash", script, *CFG["overrides"], *extra]
    if not execute:
        return 0

    proc = subprocess.Popen(
        cmd,
        cwd=ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


def persist_flush(label: str) -> None:
    drive = os.environ.get("LLM4REC_DRIVE")
    if not drive:
        return
    stamp = Path(drive) / "_persist" / f"{label}.ok"
    stamp.parent.mkdir(parents=True, exist_ok=True)
    stamp.write_text(f"{label}\n", encoding="utf-8")
    try:
        os.sync()
    except OSError:
        pass
    print(f"[persist] flushed {label} → {drive}")


def require_ok(code: int, what: str) -> None:
    if code != 0:
        raise RuntimeError(f"{what} 退出码 {code}")
    persist_flush(what)


In [5]:
# prepare.sh: download,data,embed,sid（MovieLens 不跑 bm25）
# 旧 K=512 SID 不能复用；sid_ready() 只认当前 codebook（256）。
need_data = not (CFG["skip_if_done"] and processed_ready())
need_sid = not (CFG["skip_if_done"] and sid_ready())
if need_data or need_sid:
    steps = "download,data,embed,sid" if need_data else "sid"
    if need_sid and not need_data:
        print(f"[prepare] 需要 codebook={_wanted_sid_codebook()} 的 SID，重建 SID")
    require_ok(
        run_script(
            "prepare.sh",
            env_extra={"STEPS": steps, "GPUS": "0"},
        ),
        "prepare.sh",
    )
else:
    print("[prepare] skip_if_done：processed + SID 已就绪")


In [6]:
# SFT。force_sft=True 或只有中间产物（没有 sft/final）时会重训。
# 旧 074107 与新 recipe / K=256 不兼容，CFG.force_sft 必须先为 True。
sft_ckpt = resolve_sft_final()
force_sft = bool(CFG.get("force_sft"))
if force_sft:
    print("[sft] force_sft=True，忽略已有产物，按官方 mix + Alpaca 重新训练")
    CFG["sft_checkpoint"] = ""
    sft_ckpt = None
if not (CFG["skip_if_done"] and sft_ckpt is not None):
    require_ok(run_script("run.sh", env_extra={"STAGES": "sft"}), "SFT")
    sft_ckpt = resolve_sft_final()
elif sft_ckpt is not None:
    print(f"[sft] skip_if_done：已有完整权重 {sft_ckpt}")


[sft] skip_if_done：已有完整权重 /content/llm4rec-bias-Integrated/runs/movielens_ml-100k/minionerec/qwen2.5-0.5b-instruct/seed_42/20260822_074107/sft/final


In [7]:
# eval SFT：RESUME_FROM 必须指向已扩词表的 sft/best 或 sft/final
sft_ckpt = resolve_sft_eval_ckpt()
if sft_ckpt is None:
    raise FileNotFoundError("找不到 sft/final，先跑上一格。")

eval_sft = find_eval_json(sft_ckpt)
force_eval = bool(CFG.get("force_eval"))
if eval_sft is not None and not force_eval and CFG["skip_if_done"]:
    attach_eval_json(sft_ckpt, eval_sft)
    print(f"[eval] 复用已有结果 {eval_sft}")
    print(f"[eval] checkpoint {sft_ckpt}")
else:
    if force_eval:
        print(f"[eval] force_eval=True，重评 {sft_ckpt}")
    require_ok(
        run_script("run.sh", env_extra={"STAGES": "eval", "RESUME_FROM": str(sft_ckpt)}),
        "SFT eval",
    )
    eval_sft = find_eval_json(sft_ckpt)
    if eval_sft is not None:
        attach_eval_json(sft_ckpt, eval_sft)
        print(f"[eval] 指标 → {eval_sft}")



[eval] force_eval=True，重评 /content/llm4rec-bias-Integrated/runs/movielens_ml-100k/minionerec/qwen2.5-0.5b-instruct/seed_42/20260822_074107/sft/best
WARN: 找不到 conda，跳过环境切换（CONDA_ENV=bias）
════════════════════════════════════════════════════════════
 实验     : minionerec_qwen05b_amazon
 阶段     : eval
 显卡     : 0  (n=1)
 并行     : 单卡
 wandb    : offline / llm4rec-bias
 覆盖项   : data.name=movielens data.variant=ml-100k data.category=ml-100k data.raw_dir=data/raw/movielens data.processed_dir=data/processed/movielens data.category_prompt=movies data.item_text_fields=[title,categories] data.item_text_max_chars=256 train.sft.global_batch_size=64 train.sft.per_device_batch_size=32 train.sft.preferred_per_device_batch_size=32 train.sft.eval_steps=200 train.sft.load_best_model_at_end=true train.transition.batch_size=4096 train.distill.global_batch_size=128 train.distill.per_device_batch_size=2 train.distill.preferred_per_device_batch_size=2 train.distill.catalog_chunk_size=1024 train.distill.eval_st

In [8]:
# 后训练：从 SFT 接着跑 transition + distill，不要重跑 sft
sft_ckpt = resolve_sft_final()
if sft_ckpt is None:
    raise FileNotFoundError("找不到 sft/final，后训练必须从 SFT 接着来。")

distill_ckpt = resolve_distill_final()
if not (CFG["skip_if_done"] and distill_ckpt is not None):
    require_ok(
        run_script(
            "run.sh",
            env_extra={"STAGES": "transition,distill", "RESUME_FROM": str(sft_ckpt)},
        ),
        "transition+distill",
    )
    distill_ckpt = resolve_distill_final()


WARN: 找不到 conda，跳过环境切换（CONDA_ENV=bias）
════════════════════════════════════════════════════════════
 实验     : minionerec_qwen05b_amazon
 阶段     : transition,distill
 显卡     : 0  (n=1)
 并行     : 单卡
 wandb    : offline / llm4rec-bias
 覆盖项   : data.name=movielens data.variant=ml-100k data.category=ml-100k data.raw_dir=data/raw/movielens data.processed_dir=data/processed/movielens data.category_prompt=movies data.item_text_fields=[title,categories] data.item_text_max_chars=256 train.sft.global_batch_size=64 train.sft.per_device_batch_size=32 train.sft.preferred_per_device_batch_size=32 train.sft.eval_steps=200 train.sft.load_best_model_at_end=true train.transition.batch_size=4096 train.distill.global_batch_size=128 train.distill.per_device_batch_size=2 train.distill.preferred_per_device_batch_size=2 train.distill.catalog_chunk_size=1024 train.distill.eval_steps=200 hardware.precision=bf16 hardware.gradient_checkpointing=true hardware.activation_checkpointing=auto hardware.memory=auto evalu

KeyboardInterrupt: 

In [ ]:
# eval distill：必须 RESUME_FROM=distill/final 或 best
distill_ckpt = resolve_distill_final()
if distill_ckpt is None:
    raise FileNotFoundError("找不到 distill/final。蒸馏还在跑的话等 console.log 出现 [distill] 完成。")

eval_distill = find_eval_json(distill_ckpt)
if eval_distill is not None and not bool(CFG.get("force_eval")) and CFG["skip_if_done"]:
    attach_eval_json(distill_ckpt, eval_distill)
    print(f"[eval] 复用 distill 结果 {eval_distill}")
else:
    rc = run_script(
        "run.sh",
        env_extra={"STAGES": "eval", "RESUME_FROM": str(distill_ckpt)},
    )
    require_ok(rc, "distill eval")
    eval_distill = find_eval_json(distill_ckpt)
    if eval_distill is not None:
        attach_eval_json(distill_ckpt, eval_distill)


In [ ]:
def _load_eval(path: Path) -> dict[str, Any]:
    obj = json.loads(path.read_text())
    return {"metrics": obj.get("metrics") or obj}


def _cell(metrics: dict[str, Any], key: str, width: int = 12) -> str:
    value = metrics.get(key)
    if value is None:
        return f"{'—':>{width}}"
    return f"{float(value):{width}.4f}"


def _print_table(title: str, keys: list[str]) -> None:
    print(title)
    print(f"{'tag':<10} " + " ".join(f"{k:>12}" for k in keys))
    for tag, rec in rows:
        print(f"{tag:<10} " + " ".join(_cell(rec["metrics"], k) for k in keys))
    print()


rows = []
sft = resolve_sft_eval_ckpt() or resolve_sft_final()
if sft is not None:
    p = find_eval_json(sft)
    if p is not None:
        rec = _load_eval(p)
        rec["path"] = str(p)
        rec["checkpoint"] = str(sft)
        rows.append(("SFT", rec))
        print(f"SFT eval  {p}")
        print(f"SFT ckpt  {sft}")

dist = resolve_distill_final()
if dist is not None:
    p = find_eval_json(dist)
    if p is not None:
        rec = _load_eval(p)
        rec["path"] = str(p)
        rec["checkpoint"] = str(dist)
        rows.append(("distill", rec))
        print(f"distill eval  {p}")

if not rows:
    print("还没有找到 eval_1.json。先跑 SFT eval 那一格。")

_print_table(
    "accuracy",
    ["hr@1", "hr@10", "ndcg@10", "mrr", "valid_rate"],
)
_print_table(
    "bias / popularity & exposure",
    ["pop_lift@1", "pop_lift@10", "delta_gap", "exposure_gini", "exposure_entropy", "coverage@10"],
)
_print_table(
    "bias / long-tail & shortcut",
    [
        "hr@10_head",
        "hr@10_mid",
        "hr@10_tail",
        "tier_gap",
        "hr_ips@10",
        "ndcg_ips@10",
        "history_copy_rate",
        "top1_concentration",
        "top1_distinct",
    ],
)
